# Vicky — run the LINE bot from Jupyter

Run the cells from top to bottom.

1. **Setup** — install packages (first time only) and load `.env`
2. **Start** — the bot runs in the background, so you can keep using the notebook
3. **Public URL** — optional ngrok tunnel; it also sets the LINE webhook URL for you
4. **Try it here** — validate index codes or files and ask Vicky questions without LINE
5. **Stop** — shut the bot down

The bot stops when the notebook kernel stops or the computer sleeps.

## 1. Setup

In [ ]:
# First time only: install the bot's packages into this notebook's Python
%pip install -q -r requirements.txt

In [ ]:
import logging, os, sys

# The notebook must run from the project folder (where app.py and .env are)
print("Folder:", os.getcwd())
assert os.path.exists("app.py"), "Open this notebook from the AniChan_RealAI folder"
sys.path.insert(0, os.getcwd())

from dotenv import load_dotenv
load_dotenv(override=True)

# Show the bot's log lines under the cells
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("werkzeug").setLevel(logging.WARNING)

for key in ["LINE_CHANNEL_ACCESS_TOKEN", "LINE_CHANNEL_SECRET", "GEMINI_API_KEY", "ADMIN_USER_IDS"]:
    print(f"{key:28} {'set' if os.getenv(key) else 'MISSING'}")

## 2. Start the bot

In [ ]:
import threading
from werkzeug.serving import make_server

import app as vicky

PORT = int(os.getenv("PORT", "8080"))

# Re-running this cell restarts the server instead of failing with "port in use"
if "server" in globals():
    server.shutdown()

server = make_server("0.0.0.0", PORT, vicky.app, threaded=True)
threading.Thread(target=server.serve_forever, daemon=True).start()
print(f"Vicky is running on http://localhost:{PORT}  (reference loaded: {vicky.obk_validator.get_master() is not None})")

## 3. Public URL for LINE (optional)

LINE needs a public https address. If you already use ngrok or Cloud Run, skip this cell.
Otherwise put `NGROK_AUTHTOKEN=...` in `.env` (free at https://dashboard.ngrok.com) and run it.
It opens a tunnel, tells Vicky her public address for download links, and sets the LINE webhook URL.

In [ ]:
import requests

if os.getenv("NGROK_AUTHTOKEN"):
    %pip install -q pyngrok
    from pyngrok import conf, ngrok
    conf.get_default().auth_token = os.getenv("NGROK_AUTHTOKEN")
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    public_url = ngrok.connect(PORT, "http").public_url.replace("http://", "https://")
    vicky.PUBLIC_BASE_URL = public_url

    r = requests.put("https://api.line.me/v2/bot/channel/webhook/endpoint",
                     headers=vicky.line_headers(), json={"endpoint": public_url + "/"}, timeout=10)
    print("Public URL:", public_url)
    print("LINE webhook updated" if r.ok else f"Could not update the LINE webhook: {r.status_code} {r.text}")
else:
    print("NGROK_AUTHTOKEN is not set; skipping. Set the webhook URL in LINE Developers yourself.")

## 4. Try it here (no LINE needed)

In [ ]:
# Validate index codes with the same program the bot uses
codes = ["C3A-001-ME01-AC-AHUS-000AHU-001", "C3A-001-ME01-AC-AHUX-000AHU-002"]
master = vicky.obk_validator.get_master()
print(vicky.obk_validator.format_report([vicky.obk_validator.check_code(c, master) for c in codes], "th"))

In [ ]:
# Validate a raw data file on this computer and build the summary table
path = r"Raw Data.xlsx"          # change to your file

if os.path.exists(path):
    with open(path, "rb") as f:
        result, token = vicky.obk_files.validate_file(os.path.basename(path), f.read())
    print(vicky.obk_files.format_summary(result, os.path.basename(path), "th"))
    if not result.get("skipped"):
        summary = vicky.obk_files.build_summary_workbook(result, os.path.basename(path), "th")
        print("\nFiles:", *result["outputs"], summary, sep="\n  ")
else:
    print("File not found:", path)

In [ ]:
# Ask Vicky a question the way the group would (uses Gemini and the tools, nothing is sent to LINE)
def ask(text, chat="notebook"):
    vicky.user_langs["notebook-user"] = vicky.detect_language(text) or "th"
    ctx = {"chat_id": chat, "user_id": "notebook-user", "base_url": vicky.PUBLIC_BASE_URL or f"http://localhost:{PORT}"}
    reply, links = vicky.generate_response(chat, [vicky.types.Part.from_text(text=text)], text, "notebook-user", ctx)
    vicky.remember(chat, text, reply)
    print(reply)
    for line in links["docs"] + links["files"]:
        print(line)

ask("index codes มีกี่ส่วน อะไรบ้าง")

In [ ]:
ask("ขอตาราง Raw file summary หน่อย")

## 5. Stop the bot

In [ ]:
server.shutdown()
try:
    from pyngrok import ngrok
    ngrok.kill()
except ImportError:
    pass
print("Vicky stopped")